In [1]:
import pandas as pd
import numpy as np

In [2]:
!python -m pip install --upgrade pip
!python -m pip install autogluon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is still looking at mul

In [3]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [20]:
df = pd.read_csv('filtered_data_concat.csv', index_col=0)
train_data = TabularDataset(df)
train_data

,scan_ratio,dst_density,port_density,conn_per_src,dst_per_time,pkt_ratio,burst_10s,unique_ports,label
0,0.066014,0.602491,0.066014,991061,22115.037037,3.00,26,65424,1
1,0.066014,0.602491,0.066014,991061,22115.037037,1.00,26,65424,1
2,0.066014,0.602491,0.066014,991061,22115.037037,1.00,26,65424,1
3,0.066014,0.602491,0.066014,991061,22115.037037,3.00,26,65424,1
4,0.066014,0.602491,0.066014,991061,22115.037037,1.00,26,65424,1
...,...,...,...,...,...,...,...,...,...
4769019,0.000006,0.000020,0.000006,3602157,18.250000,1.00,3,20,1
4769020,0.000006,0.000020,0.000006,3602157,24.333333,1.25,2,20,1
4769021,0.000006,0.000020,0.000006,3602157,9.125000,1.00,7,20,0
4769022,0.000006,0.000020,0.000006,3602157,18.250000,1.00,3,20,0


In [21]:
label = 'label'
train_data[label].describe()

,label
count,4.769024e+06
mean,8.997258e-01
std,3.003653e-01
min,0.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,1.000000e+00


In [26]:
class_weights = {1: 1, 0: 10}

predictor = TabularPredictor(label=label, eval_metric='f1').fit(
    train_data,
    hyperparameters={
        'GBM': {'scale_pos_weight': 10},
        'RF': {'class_weight': 'balanced'},
        'XGB': {'scale_pos_weight': 10},
    },
    presets='best_v150'
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260424_134907"
Preset alias specified: 'best_v150' maps to 'best_quality_v150'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       6.97 GB / 12.67 GB (55.0%)
Disk Space Avail:   74.41 GB / 107.72 GB (69.1%)
Presets specified: ['best_v150']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/content/AutogluonModels/ag-20260424_134907"
Train Data Rows:    4769024
Train Data Columns: 8
Label Column:       label
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-va

In [27]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM_BAG_L1,0.949837,f1,30.619729,610.706799,30.619729,610.706799,1,True,1
1,WeightedEnsemble_L2,0.949837,f1,31.729704,618.063709,1.109975,7.356910,2,True,3
2,XGBoost_BAG_L1,0.949635,f1,6.047456,289.591131,6.047456,289.591131,1,True,2


## Testing

In [31]:
predictor.evaluate(test_data, silent=True)

{'f1': 0.6691755878567168,
 'accuracy': 0.5032786952860911,
 'balanced_accuracy': np.float64(0.6636211143877879),
 'mcc': np.float64(0.02169579949672508),
 'roc_auc': np.float64(0.9004600503174969),
 'precision': 0.999615446843733,
 'recall': 0.502925142540975}

In [17]:
df_test = pd.read_csv('/content/test_data.csv', index_col=0)

/tmp/ipykernel_12384/4000806388.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('/content/test_data.csv', index_col=0)


In [18]:
# function untuk memudahkan hidup

def preprocess_data(df):
    for col in df.columns:
        if df[col].dtype == "float64":
            df[col] = df[col].fillna(0)
        else:
            df[col] = df[col].fillna("unknown")

    df["id.orig_p"] = df["id.orig_p"].astype(int)
    df["id.resp_p"] = df["id.resp_p"].astype(int)
    df["local_orig"] = df["local_orig"].astype(int)
    df["local_resp"] = df["local_resp"].astype(int)
    df = df[df["duration"] >= 0]
    df = df[df["orig_bytes"] >= 0]
    df['label'] = df['label'].map({'Malicious': 1, 'Benign': 0}).astype('category')
    df["ts"] = pd.to_datetime(df["ts"], unit="s")

    df["bytes_total"] = df["orig_bytes"] + df["resp_bytes"]
    df["bytes_ratio"] = df["orig_bytes"] / (df["resp_bytes"] + 1)
    df["pkts_total"] = df["orig_pkts"] + df["resp_pkts"]
    df["bytes_per_pkt"] = df["bytes_total"] / (df["pkts_total"] + 1)
    df["flow_rate"] = df["bytes_total"] / (df["duration"] + 1e-6)
    df["pkt_ratio"] = df["orig_pkts"] / (df["resp_pkts"] + 1)
    freq = df["service"].value_counts()
    df["service_freq"] = df["service"].map(freq)
    df["unique_ports"] = df.groupby("id.orig_h")["id.resp_p"].transform("nunique")
    df["unique_dst"] = df.groupby("id.orig_h")["id.resp_h"].transform("nunique")

    df["conn_per_src"] = df.groupby("id.orig_h")["uid"].transform("count")
    df["unique_dst"] = df.groupby("id.orig_h")["id.resp_h"].transform("nunique")
    df["scan_ratio"] = df["unique_ports"] / (df["conn_per_src"] + 1)
    df["time_bin"] = df["ts"].dt.floor("10s")
    df["burst_10s"] = df.groupby(["id.orig_h","time_bin"])["uid"].transform("count")
    df["burst_ratio"] = df["burst_10s"] / (df["conn_per_src"] + 1)
    df["failed_conn"] = df["conn_state"].isin(["S0", "REJ"]).astype(int)
    df["fail_ratio"] = df.groupby("id.orig_h")["failed_conn"].transform("mean")
    df["dst_per_time"] = df["unique_dst"] / (df["burst_10s"] + 1)
    df["port_density"] = df["unique_ports"] / (df["conn_per_src"] + 1)
    df["dst_density"] = df["unique_dst"] / (df["conn_per_src"] + 1)

    for col in ["duration", "pkts_total", "orig_pkts"]:
        df[col] = np.log1p(df[col])

    drop_cols = [
        "id.resp_p",
        "id.orig_p",
        "orig_ip_bytes",
        "uid"
    ]

    df = df.drop(columns=drop_cols)
    df = df.dropna(subset=['label'])

    # Select the top N most important features (e.g., top 5)
    sorted_importances = pd.read_csv('important_features.csv', index_col=0)
    top_n_features = sorted_importances.head(8).index.to_list()

    # Ensure 'label' is also included for further analysis
    features_to_keep = top_n_features + ['label']
    df_filtered = df[features_to_keep]

    return df_filtered

In [19]:
df_test_processed = preprocess_data(df_test)

In [30]:
test_data = TabularDataset(df_test_processed)

y_pred = predictor.predict(test_data.drop(columns=[label]))
y_pred.head()

,label
0,1
1,1
2,1
3,1
4,1
